# Cohort Analysis
Deep-dive slices: mobile vs desktop, new vs established accounts, and geo-region risk distribution.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import types

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/scratch/fkalghan/circuit_discovery_and_supression/graphs_ai_psych/identity-risk-engine")

SRC_ROOT = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from identity_risk_engine.synthetic_data_generator import generate_synthetic_login_data, train_test_time_split
    from identity_risk_engine.composite_scorer import CompositeRiskScorer
except Exception as pkg_exc:
    pkg_name = "identity_risk_engine"
    pkg_dir = SRC_ROOT / pkg_name

    if pkg_name not in sys.modules:
        pkg = types.ModuleType(pkg_name)
        pkg.__path__ = [str(pkg_dir)]
        sys.modules[pkg_name] = pkg

    def _load_submodule(name: str):
        full = f"{pkg_name}.{name}"
        if full in sys.modules:
            return sys.modules[full]
        spec = importlib.util.spec_from_file_location(full, pkg_dir / f"{name}.py")
        module = importlib.util.module_from_spec(spec)
        sys.modules[full] = module
        assert spec and spec.loader
        spec.loader.exec_module(module)
        return module

    synthetic_mod = _load_submodule("synthetic_data_generator")
    composite_mod = _load_submodule("composite_scorer")

    generate_synthetic_login_data = synthetic_mod.generate_synthetic_login_data
    train_test_time_split = synthetic_mod.train_test_time_split
    CompositeRiskScorer = composite_mod.CompositeRiskScorer

from dashboard.metrics_dashboard import build_metrics_dashboard, threshold_metrics_table


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

df = generate_synthetic_login_data(num_users=360, num_sessions=16000, attack_ratio=0.21, seed=29)
train_df, test_df = train_test_time_split(df, test_ratio=0.25)
model = CompositeRiskScorer(random_state=29)
model.fit(train_df, target_col='is_attack')

analysis_df = test_df.copy()
analysis_df['risk_score'] = model.predict_proba(analysis_df)[:, 1]
analysis_df['platform_group'] = np.where(analysis_df['device_type'].str.contains('mobile', case=False, na=False), 'mobile', 'desktop')
analysis_df['account_cohort'] = np.where(analysis_df['account_age_days'] < 7, 'new(<7d)', 'established')
analysis_df.head()


In [ ]:
platform_summary = (
    analysis_df.groupby('platform_group', as_index=False)
    .agg(
        sessions=('session_id', 'size'),
        attack_rate=('is_attack', 'mean'),
        avg_risk_score=('risk_score', 'mean'),
    )
    .sort_values('avg_risk_score', ascending=False)
)
platform_summary

In [ ]:
px.bar(
    platform_summary,
    x='platform_group',
    y='avg_risk_score',
    color='attack_rate',
    title='Mobile vs Desktop: Average Risk Score and Attack Rate',
    text='sessions',
    color_continuous_scale='Reds',
)

In [ ]:
cohort_summary = (
    analysis_df.groupby('account_cohort', as_index=False)
    .agg(
        sessions=('session_id', 'size'),
        attack_rate=('is_attack', 'mean'),
        avg_risk_score=('risk_score', 'mean'),
        high_value_rate=('high_value_action', 'mean'),
    )
    .sort_values('account_cohort')
)
cohort_summary

In [ ]:
px.scatter(
    cohort_summary,
    x='attack_rate',
    y='avg_risk_score',
    size='sessions',
    color='account_cohort',
    hover_data=['high_value_rate'],
    title='New vs Established Accounts: Risk/Attack Tradeoff',
)

In [ ]:
region_summary = (
    analysis_df.groupby('region', as_index=False)
    .agg(
        sessions=('session_id', 'size'),
        attack_rate=('is_attack', 'mean'),
        avg_risk_score=('risk_score', 'mean'),
    )
    .sort_values(['avg_risk_score', 'attack_rate'], ascending=False)
)
region_summary

In [ ]:
px.bar(
    region_summary,
    x='region',
    y='avg_risk_score',
    color='attack_rate',
    title='Geo-Region Breakdown: Average Risk Score',
    text='sessions',
    color_continuous_scale='Oranges',
)